# Анализ рынка недвижимости и прогнозирование цен

**Цель проекта:** Исследовать геометрические параметры домов (`dim_1`, `dim_2`), провести кластеризацию объектов, настроить классификатор уровней комфорта и построить регрессионную модель для оценки стоимости недвижимости.

## 1. Загрузка данных и первичный анализ
Импортируем необходимые библиотеки и загрузим наш основной датасет.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Настраиваем шрифты для графиков
plt.rc("font", size=14)

# Загружаем данные
houses = pd.read_csv("../data/1.4_houses.csv")

# Выводим первые 5 строк датасета, чтобы убедиться в успешной загрузке
houses.head()

,dim_1,dim_2,level,price
0,29,28,luxury,2212.0
1,28,29,luxury,2203.0
2,6,9,basic,254.0
3,5,9,basic,242.0
4,6,6,basic,195.0


## 2. Разведочный анализ данных (EDA) и разделение выборки
*Здесь мы изучим распределение данных и подготовим их к обучению моделей.*

In [2]:
# Посмотрим на типы данных и общую информацию о датасете
print("--- Информация о датасете ---")
houses.info()

# Проверим наличие пропущенных значений в каждой колонке
print("\n--- Проверка на пропущенные значения ---")
print(houses.isna().sum())

# Выведем основные статистические метрики для числовых признаков
print("\n--- Статистическое описание числовых признаков ---")
houses.describe()

--- Информация о датасете ---
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   dim_1   100 non-null    int64  
 1   dim_2   100 non-null    int64  
 2   level   100 non-null    str    
 3   price   100 non-null    float64
dtypes: float64(1), int64(2), str(1)
memory usage: 3.8 KB

--- Проверка на пропущенные значения ---
dim_1    0
dim_2    0
level    0
price    0
dtype: int64

--- Статистическое описание числовых признаков ---


,dim_1,dim_2,price
count,100.000000,100.000000,100.000000
mean,16.400000,16.110000,1010.970000
std,9.340906,9.105326,801.572879
min,5.000000,5.000000,150.000000
25%,7.000000,7.000000,244.250000
50%,17.000000,14.000000,808.000000
75%,27.000000,27.000000,1992.500000
max,29.000000,29.000000,2285.000000


## 3. Разделение данных и обучение моделей

Чтобы объективно оценить качество работы наших моделей, мы разделим данные на две части:
* **Обучающая выборка (Train)** — 80% данных для настройки моделей.
* **Тестовая выборка (Test)** — 20% данных, которые модель не видела, для финальной проверки.

Начнем с задачи **Классификации** (предсказание категориальной целевой переменной `level` по признакам `dim_1` и `dim_2`).

In [3]:
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# 1. Выделяем признаки и целевую переменную
X_clf = houses[["dim_1", "dim_2"]]
y_clf = houses["level"]

# 2. Разделяем на train и test выборки
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)

# 3. Создаем и обучаем классификатор на тренировочных данных
cl = DecisionTreeClassifier().fit(X_train_clf, y_train_clf)

# 4. Делаем предсказание на тестовых данных, которые модель еще не видела
y_pred_clf = cl.predict(X_test_clf)

# 5. Выводим метрики качества (Precision, Recall, F1-score)
print("--- Отчет о качестве классификации (Тестовая выборка) ---")
print(classification_report(y_test_clf, y_pred_clf))

--- Отчет о качестве классификации (Тестовая выборка) ---
              precision    recall  f1-score   support

       basic       1.00      1.00      1.00        10
      luxury       1.00      1.00      1.00         6
      medium       1.00      1.00      1.00         4

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20

